In [1]:
import pandas as pd

In [8]:
df = pd.read_csv("../outputs/control/20261004_234217/results.csv")

In [29]:
df

,model,n_folds_ok,accuracy_mean,accuracy_std,auc_mean,auc_std,train_time_mean_s,train_time_std_s,inference_time_mean_s,inference_time_std_s,...,predict_aggregation_mean_s,predict_aggregation_std_s,predict_class_remapping_mean_s,predict_class_remapping_std_s,predict_cluster_weights_mean_s,predict_cluster_weights_std_s,predict_total_predict_proba_mean_s,predict_total_predict_proba_std_s,predict_tree_predict_proba_mean_s,predict_tree_predict_proba_std_s
0,FuzzyEspecialistEnsembleClassifier,5,0.898000,0.006182,0.955979,0.006628,1.929255,0.070552,0.019848,0.000677,...,0.000284,0.000027,0.001598,0.000035,0.000094,0.000010,0.019801,0.000676,0.017053,0.000564
1,Fuzzy_m1.5,5,0.898667,0.011662,0.956312,0.007332,1.984137,0.028031,0.019560,0.000690,...,0.000283,0.000027,0.001607,0.000074,0.000089,0.000009,0.019518,0.000689,0.016830,0.000599
2,Fuzzy_m1.2,5,0.902000,0.013433,0.962540,0.004989,2.075677,0.157819,0.018665,0.000510,...,0.000307,0.000023,0.001584,0.000061,0.000094,0.000013,0.018629,0.000510,0.015939,0.000415
3,GMMEspecialistEnsembleClassifier,5,0.862667,0.020155,0.949765,0.004263,2.032677,0.091592,0.018423,0.000361,...,0.000263,0.000018,0.001573,0.000043,0.000080,0.000006,0.018383,0.000361,0.015786,0.000337
4,FuzzyKMedoidsEspecialistEnsembleClassifier,5,0.903333,0.006992,0.954668,0.005570,1.815987,0.250834,0.019042,0.000292,...,0.000282,0.000036,0.001568,0.000019,0.000089,0.000006,0.019000,0.000290,0.016350,0.000284
5,PossibilisticCMeansEspecialistEnsembleClassifier,5,0.894667,0.009092,0.949823,0.006967,1.917791,0.049183,0.019520,0.000466,...,0.000323,0.000113,0.001615,0.000054,0.000102,0.000027,0.019479,0.000466,0.016704,0.000281
6,FastKSelectionEnsembleClassifier,5,0.901333,0.010873,0.956126,0.006836,1.673361,0.046310,0.020299,0.001342,...,0.000287,0.000023,0.001655,0.000117,0.000089,0.000010,0.020256,0.001342,0.017496,0.001177
7,DecisionTree,5,0.848667,0.018451,0.848660,0.018346,0.010670,0.000390,0.000322,0.000054,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Bagging,5,0.898667,0.009568,0.955672,0.008304,0.420810,0.014475,0.009934,0.000864,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,Boosting,5,0.846667,0.009661,0.923750,0.009520,0.163495,0.004111,0.008958,0.000099,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
df[['model', 'train_time_mean_s', 'train_time_std_s', 'inference_time_mean_s', 'inference_time_std_s', 'fit_find_best_k_mean_s', 'fit_find_best_k_std_s']].iloc[[0,6]]

,model,train_time_mean_s,train_time_std_s,inference_time_mean_s,inference_time_std_s,fit_find_best_k_mean_s,fit_find_best_k_std_s
0,FuzzyEspecialistEnsembleClassifier,1.929255,0.070552,0.019848,0.000677,0.331493,0.007267
6,FastKSelectionEnsembleClassifier,1.673361,0.046310,0.020299,0.001342,0.056158,0.004216


In [27]:
from itertools import combinations
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind_from_stats

def pairwise_tests(df, metrics=("accuracy", "auc", "train_time", "inference_time",
                                "fit_find_best_k", "fit_train_classifiers"), alpha=0.05):
    rows = []
    for metric in metrics:
        # column naming: accuracy_mean / train_time_mean_s / fit_find_best_k_mean_s ...
        mean_col = next(c for c in df.columns if c.startswith(metric + "_mean"))
        std_col  = mean_col.replace("_mean", "_std")

        for a, b in combinations(df.index, 2):
            ra, rb = df.loc[a], df.loc[b]
            na, nb = int(ra["n_folds_ok"]), int(rb["n_folds_ok"])
            if min(na, nb) < 2 or pd.isna(ra[mean_col]) or pd.isna(rb[mean_col]):
                continue
            # stored std is np.std (ddof=0) -> convert to sample std (ddof=1)
            sa = ra[std_col] * np.sqrt(na / (na - 1))
            sb = rb[std_col] * np.sqrt(nb / (nb - 1))
            t, p = ttest_ind_from_stats(ra[mean_col], sa, na,
                                        rb[mean_col], sb, nb, equal_var=False)
            rows.append({"metric": metric, "model_a": ra["model"], "model_b": rb["model"],
                         "mean_a": ra[mean_col], "mean_b": rb[mean_col],
                         "diff": ra[mean_col] - rb[mean_col], "t": t, "p": p})

    out = pd.DataFrame(rows)
    # Holm correction within each metric (many pairs -> many false positives otherwise)
    out["p_holm"] = out.groupby("metric")["p"].transform(
        lambda p: np.minimum(1, (p.rank(method="first").rsub(len(p) + 1) * p)
                             .reindex(p.sort_values().index).cummax().reindex(p.index)))
    out["significant"] = out["p_holm"] < alpha
    return out

df = pd.read_csv("../outputs/control/20261004_234217/results.csv")
print(pairwise_tests(df.iloc[[0,6]]).to_string(index=False))

               metric                            model_a                          model_b   mean_a   mean_b      diff         t            p       p_holm  significant
             accuracy FuzzyEspecialistEnsembleClassifier FastKSelectionEnsembleClassifier 0.898000 0.901333 -0.003333 -0.533002 6.121872e-01 6.121872e-01        False
                  auc FuzzyEspecialistEnsembleClassifier FastKSelectionEnsembleClassifier 0.955979 0.956126 -0.000147 -0.030808 9.761780e-01 9.761780e-01        False
           train_time FuzzyEspecialistEnsembleClassifier FastKSelectionEnsembleClassifier 1.929255 1.673361  0.255894  6.064328 5.355892e-04 5.355892e-04         True
       inference_time FuzzyEspecialistEnsembleClassifier FastKSelectionEnsembleClassifier 0.019848 0.020299 -0.000451 -0.600646 5.703784e-01 5.703784e-01        False
      fit_find_best_k FuzzyEspecialistEnsembleClassifier FastKSelectionEnsembleClassifier 0.331493 0.056158  0.275336 65.542794 2.578703e-10 2.578703e-10         Tru